# WARNING: Still need to check the descriptions here, some are still incorrect. Code should be fine

# Dataset Statistics: Identification Drift over Time

Characterises how the number of identified precursors and proteins changes as a function of time at two scales:

1. **Inter-day variability** — identification counts per condition, broken down by acquisition day per lab. Reveals whether instrument performance is consistent between measurement sessions or degrades/improves over the course of the study.
2. **Long-term stability** — normalised identification counts plotted against days elapsed since a lab's first acquisition. Summarised as median ± IQR across all cells of each condition, enabling direct cross-lab comparison of multi-day performance trends.
3. **Intra-day drift** — within a single acquisition day, how identification depth changes with injection order. Time within each day is normalised to [0, 1] and a linear regression is fitted per lab × condition. Statistically significant negative slopes indicate that later-injected cells yield fewer identifications — a classic sign of column overloading, carry-over, or electrospray instability.

Throughout this notebook, samples flagged as `completeness_outlier` (failed injections) are excluded to prevent near-empty runs from distorting drift estimates.

**Inputs** (`Data/AnnData/`)

| File | Description |
|---|---|
| `adata_precursor.h5ad` | Precursor-level quantification (samples × precursors) |
| `adata_protein.h5ad` | Protein-level quantification (samples × protein groups) |

**Outputs** (`Figures/Figure_4/Drift/`)

All figures are saved as png, svg, and pdf.

| Subfolder | File pattern | Description |
|---|---|---|
| `Across_days/` | `{lab}_precursor_DriftByDay` | Per-lab precursor counts, split by acquisition day |
| `Across_days/` | `{lab}_protein_DriftByDay` | Per-lab protein counts, split by acquisition day |
| `Within_day/` | `Overview_precursor_DriftWithinDay` | Long-term precursor stability, all labs |
| `Within_day/` | `Overview_protein_DriftWithinDay` | Long-term protein stability, all labs |
| `Correlation/` | `{lab}_drift_precursor` | Per-lab intra-day precursor drift regression |
| `Correlation/` | `{lab}_drift_protein` | Per-lab intra-day protein drift regression |
| `Correlation/` | `drift_significance` | Heatmap of BH-corrected drift p-values |

---

## Imports

In [1]:

import os
from tqdm import tqdm
import numpy as np
import pandas as pd
from pathlib import Path
import anndata

from statsmodels.stats.multitest import multipletests

from SCI_py.constants.maps import LABS
from SCI_py.metrics.drift import compute_drift, fit_drift_stats
from SCI_py.plot.drift import plot_drift_significance, plot_drift, plot_features_by_day, plot_feature_stability

tqdm.pandas()

## 0. Path Configuration

Output directories are resolved relative to the notebook location and created automatically if absent. Both can be overridden via environment variables, which is useful for running the notebook in a CI pipeline or on a different machine without editing the notebook.

| Variable | Default | Environment override |
|---|---|---|
| `ROOT_OUT` | `../../Data` | `SCI_DATA_DIR` |
| `FIGURES_OUT` | `../../Figures` | `SCI_FIGURES_DIR` |

AnnData objects are expected under `ROOT_OUT/AnnData/`, produced by `02_Results_to_AnnData.ipynb`.

In [2]:
NOTEBOOK_DIR = Path().resolve()
ROOT_OUT      = NOTEBOOK_DIR / "../../Data"
ROOT_OUT      = ROOT_OUT.resolve()
ROOT_OUT.mkdir(parents=True, exist_ok=True)

# Allow user override via environment variable
ROOT_OUT = Path(os.environ.get("SCI_DATA_DIR", ROOT_OUT))

ROOT_OUT.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {ROOT_OUT}")

FIGURES_OUT = NOTEBOOK_DIR / "../../Figures"
FIGURES_OUT = FIGURES_OUT.resolve()
FIGURES_OUT.mkdir(parents=True, exist_ok=True)

# Overwrite with environment variable
FIGURES_OUT = Path(os.environ.get("SCI_FIGURES_DIR", FIGURES_OUT))
print(f"Figures directory: {FIGURES_OUT}")

ANNDATA_IN = os.path.join(
    ROOT_OUT,
    'AnnData'
)

Output directory: /home/sam/SCI/SCI/Data
Figures directory: /home/sam/SCI/SCI/Figures


## 1. Load Data

Load the pre-processed AnnData objects for precursor-level and protein-level quantification. Both are expected to carry `completeness_outlier` in `obs` (written by `02_QC_Metrics.ipynb`), which is used throughout this notebook to exclude failed injections before computing drift statistics.

Two condition lists are defined:

- `conditions_cell_QC` — all four biological/QC conditions (K562, HEK293T, HeLa QC, site-specific QC). Used for the per-day and stability plots where QC injections provide a stable reference.
- `condition_cell` — the two primary cell-line conditions (K562, HEK293T) only. Reserved for analyses where including QC injections would conflate biological and technical variability.

In [3]:
adata_precursor = anndata.io.read_h5ad(
    os.path.join(ANNDATA_IN, 'adata_precursor.h5ad')
)
adata_protein = anndata.io.read_h5ad(
    os.path.join(ANNDATA_IN, 'adata_protein.h5ad')
)

conditions_cell_QC = ['K562', 'HEK293T', 'Site specific QC', 'QC - Hela']
condition_cell = ['K562', 'HEK293T']

## 2. Inter-day Variability

Visualises how identification depth varies between different acquisition days for each lab. For each lab, `plot_features_by_day` produces a grouped boxplot where:

- **x-axis** — biological/QC condition (K562, HEK293T, HeLa QC, site-specific QC).
- **Colour (hue)** — individual acquisition date, ordered chronologically and coloured on a blue gradient from lightest (earliest day) to darkest (latest day). This makes it immediately apparent whether later days systematically under- or over-perform relative to earlier sessions.
- **y-axis** — identification count normalised to the per-condition maximum (`normalize=True`), expressed as a percentage. Normalisation removes the absolute-count difference between conditions and focuses attention on day-to-day consistency within each condition.

One figure is produced per lab at both precursor and protein level, saved under `Across_days/`.

In [4]:
conditions = ['K562', 'HEK293T', 'Site specific QC', 'QC - Hela']

# Single lab, by day
for plate_idx in LABS:
    plot_features_by_day(
        # Filter out failed samples
        adata=adata_precursor[
            ~adata_precursor.obs['completeness_outlier']
        ],
        count_col='n_precursors',
        lab=plate_idx,
        conditions=conditions_cell_QC,
        normalize=True,
        save_dir=os.path.join(FIGURES_OUT, 'Figure_4', 'Drift', 'Across_days'),
        filename=f'{plate_idx}_precursor_DriftByDay',
        show=False
    )
    plot_features_by_day(
        # Filter out failed samples
        adata=adata_protein[
            ~adata_protein.obs['completeness_outlier']
        ],
        count_col='n_proteins',
        lab=plate_idx,
        conditions=conditions_cell_QC,
        normalize=True,
        save_dir=os.path.join(FIGURES_OUT, 'Figure_4', 'Drift', 'Across_days'),
        filename=f'{plate_idx}_protein_DriftByDay',
        show=False
    )

## 3. Long-term Identification Stability

Summarises identification depth over the full data-collection period for all labs simultaneously. `plot_feature_stability` computes, for each lab and each condition:

1. The per-day maximum identification count (the reference for normalisation).
2. For every cell, its count as a percentage of that maximum — `{count_col}_pct`.
3. The daily median and interquartile range across all cells, plotted as a line ± shaded band against **days since first acquisition** (day 0 = the lab's first measurement session).

Each lab is drawn in its designated colour (`LAB_COLORS`). Labs that acquired all data in a single session appear as a point rather than a trajectory. A flat, overlapping bundle of lines across labs indicates stable, comparable performance throughout the study; diverging or declining trajectories flag labs where instrument performance drifted significantly over time.

One figure is produced for each quantification level (precursor and protein), with one panel per condition, saved under `Within_day/`.

In [5]:
# All labs, stability over time
plot_feature_stability(
    adata=adata_precursor[
        ~adata_precursor.obs['completeness_outlier']
    ],
    count_col='n_precursors',
    conditions=conditions_cell_QC,
    save_dir=os.path.join(FIGURES_OUT, 'Figure_4', 'Drift', 'Within_day'),
    filename='Overview_precursor_DriftWithinDay',
    show=False
)

# Using proteins instead
plot_feature_stability(
    adata=adata_protein[
        ~adata_protein.obs['completeness_outlier']
    ],
    count_col='n_proteins',
    conditions=conditions_cell_QC,
    save_dir=os.path.join(FIGURES_OUT, 'Figure_4', 'Drift', 'Within_day'),
    filename='Overview_protein_DriftWithinDay',
    show=False
)

## 4. Intra-day Drift

Measures whether identification depth decreases (or increases) systematically within a single acquisition day as a function of injection order. This is the most direct proxy for LC column conditioning, carry-over accumulation, or electrospray instability.

**Normalisation** — `compute_drift` normalises the clock time of each injection to [0, 1] within its calendar day (`time_norm = 0` = first injection of the day, `time_norm = 1` = last injection). Identification counts are similarly normalised to the per-condition, per-day maximum, yielding `{count_col}_pct`.

**Regression** — `fit_drift_stats` fits an ordinary least-squares linear regression of `{count_col}_pct ~ time_norm` per lab × condition. The slope (in % per normalised time unit) quantifies the magnitude of drift; the p-value tests whether the trend is distinguishable from noise.

**Multiple-testing correction** — raw p-values are corrected across all lab × condition combinations using the Benjamini–Hochberg (BH) procedure at α = 0.05. `plot_drift_significance` displays the corrected p-values as a colour-coded heatmap, with annotation stars for three significance tiers (*, **, ***).

`KI_Astral_2` is skipped in the per-lab scatter plots due to insufficient data for that instrument.

One scatter + regression figure is produced per lab at both precursor and protein level, plus a single combined significance heatmap, all saved under `Correlation/`.

In [6]:
# Compute once per adata
drift_precursor = compute_drift(
    adata=adata_precursor[
        ~adata_precursor.obs['completeness_outlier']
    ],
    conditions=conditions_cell_QC,
    count_col='n_precursors'
)
drift_protein = compute_drift(
    adata=adata_protein[
        ~adata_protein.obs['completeness_outlier']
    ],
    conditions=conditions_cell_QC,
    count_col='n_proteins'
)

# Plot per lab
for lab in LABS:
    # Missing data for this plate
    if lab == 'KI_Astral_2':
        continue
    
    plot_drift(
        drift_df=drift_precursor,
        lab=lab,
        count_col='n_precursors',
        conditions=conditions_cell_QC,
        save_dir=os.path.join(FIGURES_OUT, 'Figure_4', 'Drift', 'Correlation'),
        filename=f'{lab}_drift_precursor',
        show=False
    )
    plot_drift(
        drift_df=drift_protein,
        lab=lab,
        count_col='n_proteins',
        conditions=conditions_cell_QC,
        save_dir=os.path.join(FIGURES_OUT, 'Figure_4', 'Drift', 'Correlation'),
        filename=f'{lab}_drift_protein',
        show=False
    )

In [7]:
# Fit stats and correct
stats_prec = fit_drift_stats(
    drift_df=drift_precursor,
    count_col='n_precursors'
).assign(adata_type='precursor')

stats_prot = fit_drift_stats(
    drift_df=drift_protein,
    count_col='n_proteins'
).assign(adata_type='protein')

for df in (stats_prec, stats_prot):
    _, df['p_value_bh'], _, _ = multipletests(df['p_value'], method='fdr_bh')

stats_all = pd.concat([stats_prec, stats_prot])

plot_drift_significance(
    stats_df=stats_all,
    save_dir=os.path.join(FIGURES_OUT, 'Figure_4', 'Drift', 'Correlation'),
    filename='drift_significance',
    show=False
)

/home/sam/SCI/SCI/SCI_py/SCI_py/plot/drift.py:299: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  annot = pivot.applymap(lambda p:
/home/sam/SCI/SCI/SCI_py/SCI_py/plot/drift.py:299: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  annot = pivot.applymap(lambda p:
